In [17]:
# =============================================================================
# IVF Embryo Implantation Prediction – Complete Kaggle Pipeline
# =============================================================================
# Dataset structure:
#   /kaggle/input/datasets/vibhais/ivf-combi/Embryo dataset/
#   ├── Embryo dataset_2542.xlsx         ← clinical data + labels
#   └── Data set splitting/
#       └── Data set splitting/
#           ├── Pregnant/
#           │   ├── Cross-validation/    ← .jpg files named <embryo_id>.jpg
#           │   └── External-validation/
#           └── Non-pregnant/
#               ├── Cross-validation/
#               └── External-validation/
#
# Key facts from xlsx inspection:
#   - '胚胎' column = embryo_id (e.g. '0001_0') → matches image filename stem
#   - 'label' column already present (0 / 1) → no need to derive from folders
#   - 38 columns total, all clinical features identified below
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd
import joblib

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================

ROOT_DIR  = "/kaggle/input/datasets/vibhais/ivf-combi/Embryo dataset"
SPLIT_DIR = os.path.join(ROOT_DIR, "Data set splitting", "Data set splitting")
POS_DIR   = os.path.join(SPLIT_DIR, "Pregnant")
NEG_DIR   = os.path.join(SPLIT_DIR, "Non-pregnant")
XLSX_PATH = os.path.join(ROOT_DIR, "Embryo dataset_2542.xlsx")

MODEL_OUT_PATH       = "/kaggle/working/ivf_model.joblib"
SCALER_OUT_PATH      = "/kaggle/working/ivf_scaler.joblib"
FEAT_COLS_OUT_PATH   = "/kaggle/working/ivf_feature_cols.joblib"

IMAGE_SIZE   = (224, 224)
RANDOM_STATE = 42
TEST_SIZE    = 0.2

# =============================================================================
# COLUMN MAPPING — every Chinese column in the xlsx → English
# =============================================================================

COLUMN_MAP = {
    "胚胎":           "embryo_id",

    # Female hormones & blood work
    "女方年龄":        "female_age",
    "女方基础FSH":     "FSH",
    "女方基础PRL":     "PRL",
    "女方基础T":       "T",
    "女方基础P":       "P",
    "女方AMH":        "AMH",
    "女方白细胞计数":  "female_WBC",
    "女方血小板":      "female_platelet",
    "女方血细胞压积":  "female_hematocrit",
    "女方凝血常规PT":  "PT",
    "女方凝血常规APPT":"APTT",
    "女方凝血常规TT":  "TT",
    "女方D二聚体":     "D_dimer",
    "女方甲功TPO":     "TPO",
    "女方空腹血糖":    "female_glucose",
    "女方肾功尿酸":    "uric_acid",

    # Male semen & blood work
    "男方浓度":        "male_concentration",
    "男方正常形态率":  "male_normal_morphology",
    "男方PR":         "male_PR",
    "男方NP":         "male_NP",
    "男方存活率":      "male_viability",
    "男方DFI":        "male_DFI",
    "男方ALT":        "male_ALT",
    "男方AST":        "male_AST",
    "男方血肌酐":      "male_creatinine",
    "男方白细胞":      "male_WBC",
    "男方血小板":      "male_platelet",
    "男方空腹血糖":    "male_glucose",

    # Stimulation & procedure
    "女方用药Gn总天数": "stim_days",
    "女方用药HCG日P":  "HCG_P",
    "男方精液处理后NP": "processed_NP",
    "男方精液处理方法": "processing_method",
    "男方精液处理后密度":"processed_density",
    "男方精液处理后PR": "processed_PR",
    "男方精液处理后IM": "processed_IM",

    # Embryo
    "胚胎冷冻数":      "frozen_embryos",
}

# Clinical feature columns used for model training (everything except IDs & label)
CLINICAL_FEATURES = [
    "female_age", "FSH", "PRL", "T", "P", "AMH",
    "female_WBC", "female_platelet", "female_hematocrit",
    "PT", "APTT", "TT", "D_dimer", "TPO",
    "female_glucose", "uric_acid",
    "male_concentration", "male_normal_morphology", "male_PR", "male_NP",
    "male_viability", "male_DFI", "male_ALT", "male_AST",
    "male_creatinine", "male_WBC", "male_platelet", "male_glucose",
    "stim_days", "HCG_P",
    "processed_NP", "processing_method", "processed_density",
    "processed_PR", "processed_IM",
    "frozen_embryos",
]


# =============================================================================
# STEP 1 — Build image path lookup  {embryo_id: full_image_path}
# =============================================================================

def build_image_index() -> dict:
    """
    Recursively walk Pregnant/ and Non-pregnant/ and map
    filename stem → full path.  e.g. '0001_0' → '/kaggle/.../0001_0.jpg'
    """
    index = {}
    for folder in [POS_DIR, NEG_DIR]:
        if not os.path.isdir(folder):
            raise FileNotFoundError(f"Folder not found: {folder}")
        for root, _, files in os.walk(folder):
            for fname in files:
                if fname.lower().endswith(".jpg"):
                    stem = os.path.splitext(fname)[0]   # '0001_0'
                    index[stem] = os.path.join(root, fname)

    print(f"[build_image_index] Indexed {len(index)} images.")
    return index


# =============================================================================
# STEP 2 — Load and translate clinical xlsx
# =============================================================================

def load_clinical_xlsx(xlsx_path: str) -> pd.DataFrame:
    """
    Read the Excel file, rename all Chinese columns to English,
    cast numeric columns, and return a tidy DataFrame.
    """
    df = pd.read_excel(xlsx_path, dtype=str)
    df = df.rename(columns=COLUMN_MAP)

    # Cast everything except embryo_id to numeric
    for col in df.columns:
        if col != "embryo_id":
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # label must be integer
    df["label"] = df["label"].astype(int)

    print(f"[load_clinical_xlsx] {len(df)} rows | {len(df.columns)} columns")
    n_pos = (df["label"] == 1).sum()
    n_neg = (df["label"] == 0).sum()
    print(f"  Class balance — Pregnant: {n_pos} | Non-pregnant: {n_neg}")
    return df


# =============================================================================
# STEP 3 — Attach image paths; drop rows with no matching image
# =============================================================================

def attach_image_paths(df: pd.DataFrame, image_index: dict) -> pd.DataFrame:
    """
    Add an 'image_path' column by looking up embryo_id in the image index.
    Rows whose embryo_id has no image are dropped.
    """
    df = df.copy()
    df["image_path"] = df["embryo_id"].map(image_index)

    before = len(df)
    df = df.dropna(subset=["image_path"]).reset_index(drop=True)
    after  = len(df)

    print(f"[attach_image_paths] Matched {after}/{before} rows to images "
          f"(dropped {before - after} unmatched).")
    return df


# =============================================================================
# STEP 4 — Preprocess clinical features
# =============================================================================

def preprocess_clinical(df: pd.DataFrame,
                         scaler: StandardScaler = None,
                         fit_scaler: bool = True):
    """
    Median imputation for missing values + StandardScaler normalization.

    Returns
    -------
    X_scaled : np.ndarray  (n_samples, n_features)
    scaler   : fitted StandardScaler
    """
    X = df[CLINICAL_FEATURES].copy()

    # Median imputation
    for col in X.columns:
        missing = X[col].isnull().sum()
        if missing > 0:
            med = X[col].median()
            X[col].fillna(med, inplace=True)
            print(f"  Imputed {missing} missing values in '{col}' with median={med:.3f}")

    if scaler is None:
        scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X) if fit_scaler else scaler.transform(X)
    print(f"[preprocess_clinical] Clinical feature matrix: {X_scaled.shape}")
    return X_scaled, scaler


# =============================================================================
# STEP 5 — Build EfficientNetB0 feature extractor
# =============================================================================

def build_feature_extractor():
    """
    EfficientNetB0 pretrained on ImageNet, frozen weights,
    global average pooling → 1280-dim feature vector per image.
    """
    model = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        pooling="avg",
        input_shape=(*IMAGE_SIZE, 3),
    )
    model.trainable = False
    print(f"[build_feature_extractor] Output shape: {model.output_shape}")
    return model


# =============================================================================
# STEP 6 — Load and preprocess a single image
# =============================================================================

def load_image(image_path: str) -> np.ndarray:
    """
    Open JPEG, resize to 224×224, apply EfficientNet preprocessing.
    Returns array of shape (1, 224, 224, 3).
    """
    img = Image.open(image_path).convert("RGB")
    img = img.resize(IMAGE_SIZE)
    arr = np.array(img, dtype=np.float32)
    arr = preprocess_input(arr)          # scales pixel values to [-1, 1]
    return np.expand_dims(arr, axis=0)   # add batch dimension


# =============================================================================
# STEP 7 — Extract image features for entire dataset
# =============================================================================

def extract_image_features(df: pd.DataFrame, extractor) -> np.ndarray:
    """
    Run every embryo image through the CNN extractor.
    Returns feature matrix of shape (n_samples, 1280).
    Prints progress every 200 images.
    """
    features = []
    total = len(df)

    for i, (_, row) in enumerate(df.iterrows()):
        img_tensor = load_image(row["image_path"])
        feat = extractor.predict(img_tensor, verbose=0)[0]   # shape (1280,)
        features.append(feat)

        if (i + 1) % 200 == 0 or (i + 1) == total:
            print(f"  [{i + 1}/{total}] images processed...")

    features = np.array(features)
    print(f"[extract_image_features] Image feature matrix: {features.shape}")
    return features


# =============================================================================
# STEP 8 — Combine image + clinical features
# =============================================================================

def combine_features(image_feats: np.ndarray,
                     clinical_feats: np.ndarray) -> np.ndarray:
    combined = np.concatenate([image_feats, clinical_feats], axis=1)
    print(f"[combine_features] Combined feature matrix: {combined.shape}")
    return combined


# =============================================================================
# STEP 9 — Train classifier
# =============================================================================

def train_classifier(X_train: np.ndarray,
                     y_train: np.ndarray,
                     model_type: str = "xgboost"):
    """
    Train XGBoost (default) or RandomForest.
    Switch via model_type = 'random_forest'.
    """
    if model_type == "xgboost":
        clf = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
        )
    else:
        clf = RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )

    clf.fit(X_train, y_train)
    print(f"[train_classifier] Trained {model_type} on {len(X_train)} samples.")
    return clf


# =============================================================================
# STEP 10 — Evaluate
# =============================================================================

def evaluate_model(clf, X_test: np.ndarray, y_test: np.ndarray) -> dict:
    y_pred  = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    acc     = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    print("\n" + "=" * 55)
    print("MODEL EVALUATION")
    print("=" * 55)
    print(f"  Accuracy :  {acc:.4f}")
    print(f"  ROC-AUC  :  {roc_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred,
        target_names=["Non-pregnant (0)", "Pregnant (1)"]
    ))
    print("=" * 55 + "\n")
    return {"accuracy": acc, "roc_auc": roc_auc}


# =============================================================================
# STEP 11 — Save artifacts
# =============================================================================

def save_artifacts(clf, scaler):
    joblib.dump(clf,             MODEL_OUT_PATH)
    joblib.dump(scaler,          SCALER_OUT_PATH)
    joblib.dump(CLINICAL_FEATURES, FEAT_COLS_OUT_PATH)
    print(f"[save_artifacts] Model        → {MODEL_OUT_PATH}")
    print(f"[save_artifacts] Scaler       → {SCALER_OUT_PATH}")
    print(f"[save_artifacts] Feature list → {FEAT_COLS_OUT_PATH}")


def load_artifacts():
    clf           = joblib.load(MODEL_OUT_PATH)
    scaler        = joblib.load(SCALER_OUT_PATH)
    feature_cols  = joblib.load(FEAT_COLS_OUT_PATH)
    return clf, scaler, feature_cols


# =============================================================================
# INFERENCE — predict_single
# =============================================================================

def predict_single(image_path: str,
                   clinical_dict: dict,
                   extractor,
                   clf,
                   scaler: StandardScaler) -> float:
    """
    Predict pregnancy probability for ONE embryo.

    Parameters
    ----------
    image_path    : Full path to the embryo .jpg image.
    clinical_dict : Dict with English-named clinical values. Example:
                    {
                        "female_age": 31, "FSH": 4.2, "PRL": 9.0,
                        "T": 0.24, "P": 1.03, "AMH": 0.61,
                        "female_WBC": 6.24, "female_platelet": 137,
                        "female_hematocrit": 0.38, "PT": 11.2,
                        "APTT": 28.5, "TT": 14.0, "D_dimer": 0.3,
                        "TPO": 12.0, "female_glucose": 5.1,
                        "uric_acid": 280, "male_concentration": 60,
                        "male_normal_morphology": 4.5, "male_PR": 55,
                        "male_NP": 10, "male_viability": 75,
                        "male_DFI": 15, "male_ALT": 22, "male_AST": 20,
                        "male_creatinine": 85, "male_WBC": 5.5,
                        "male_platelet": 233, "male_glucose": 5.12,
                        "stim_days": 10, "HCG_P": 0.99,
                        "processed_NP": 11, "processing_method": 1,
                        "processed_density": 37.55, "processed_PR": 87.81,
                        "processed_IM": 9.44, "frozen_embryos": 2,
                    }
    extractor     : CNN feature extractor from build_feature_extractor().
    clf           : Fitted classifier.
    scaler        : Fitted StandardScaler.

    Returns
    -------
    float : Pregnancy probability in [0.0, 1.0].
    """
    # Image branch
    img_tensor  = load_image(image_path)
    image_feats = extractor.predict(img_tensor, verbose=0)       # (1, 1280)

    # Clinical branch — fill missing keys with 0
    row = {col: clinical_dict.get(col, 0.0) for col in CLINICAL_FEATURES}
    clin_df     = pd.DataFrame([row])
    clin_scaled = scaler.transform(clin_df)                      # (1, 36)

    # Combine and predict
    combined = np.concatenate([image_feats, clin_scaled], axis=1)
    proba    = float(clf.predict_proba(combined)[0, 1])

    outcome = "Pregnant ✓" if proba >= 0.5 else "Non-pregnant ✗"
    print(f"[predict_single] Probability: {proba:.4f}  →  {outcome}")
    return proba


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def main():
    print("\n" + "=" * 60)
    print("IVF EMBRYO IMPLANTATION PREDICTION — KAGGLE PIPELINE")
    print("=" * 60 + "\n")

    # 1. Index all images: {embryo_id: path}
    image_index = build_image_index()

    # 2. Load + translate clinical xlsx (label already in xlsx)
    df = load_clinical_xlsx(XLSX_PATH)

    # 3. Attach image paths; drop rows with no matching image
    df = attach_image_paths(df, image_index)

    # 4. Preprocess clinical features
    clinical_feats, scaler = preprocess_clinical(df, fit_scaler=True)

    # 5. Build CNN feature extractor
    extractor = build_feature_extractor()

    # 6. Extract image features (progress printed every 200 images)
    print("\nExtracting image features — enable GPU in Kaggle settings for speed...")
    image_feats = extract_image_features(df, extractor)

    # 7. Combine image + clinical into one feature matrix
    X = combine_features(image_feats, clinical_feats)
    y = df["label"].values

    # 8. Stratified train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

    # 9. Train
    clf = train_classifier(X_train, y_train, model_type="xgboost")

    # 10. Evaluate
    metrics = evaluate_model(clf, X_test, y_test)

    # 11. Save
    save_artifacts(clf, scaler)

    print("Pipeline complete.\n")
    return clf, scaler, extractor, metrics


# =============================================================================
# RUN
# =============================================================================

clf, scaler, extractor, metrics = main()


# =============================================================================
# EXAMPLE INFERENCE (edit values and uncomment to use)
# =============================================================================

# prob = predict_single(
#     image_path    = f"{POS_DIR}/Cross-validation/0001_0.jpg",
#     clinical_dict = {
#         "female_age": 31, "FSH": 4.2, "PRL": 9.0, "T": 0.24,
#         "P": 1.03, "AMH": 0.61, "female_WBC": 6.24,
#         "female_platelet": 137, "female_hematocrit": 0.38,
#         "PT": 11.2, "APTT": 28.5, "TT": 14.0, "D_dimer": 0.3,
#         "TPO": 12.0, "female_glucose": 5.1, "uric_acid": 280,
#         "male_concentration": 60, "male_normal_morphology": 4.5,
#         "male_PR": 55, "male_NP": 10, "male_viability": 75,
#         "male_DFI": 15, "male_ALT": 22, "male_AST": 20,
#         "male_creatinine": 85, "male_WBC": 5.5,
#         "male_platelet": 233, "male_glucose": 5.12,
#         "stim_days": 10, "HCG_P": 0.99,
#         "processed_NP": 11, "processing_method": 1,
#         "processed_density": 37.55, "processed_PR": 87.81,
#         "processed_IM": 9.44, "frozen_embryos": 2,
#     },
#     extractor     = extractor,
#     clf           = clf,
#     scaler        = scaler,
# )


IVF EMBRYO IMPLANTATION PREDICTION — KAGGLE PIPELINE

[build_image_index] Indexed 2542 images.
[load_clinical_xlsx] 2542 rows | 38 columns
  Class balance — Pregnant: 965 | Non-pregnant: 1577
[attach_image_paths] Matched 2542/2542 rows to images (dropped 0 unmatched).
[preprocess_clinical] Clinical feature matrix: (2542, 36)


I0000 00:00:1777219847.222477      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777219847.228376      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
[build_feature_extractor] Output shape: (None, 1280)

Extracting image features — enable GPU in Kaggle settings for speed...


I0000 00:00:1777219853.986439     198 service.cc:152] XLA service 0x7d040c002ec0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777219853.986478     198 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777219853.986482     198 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777219854.941087     198 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-26 16:11:01.571019: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-26 16:11:01.706488: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-26 16:11:02.464458: E external/local_xl

  [200/2542] images processed...
  [400/2542] images processed...
  [600/2542] images processed...
  [800/2542] images processed...
  [1000/2542] images processed...
  [1200/2542] images processed...
  [1400/2542] images processed...
  [1600/2542] images processed...
  [1800/2542] images processed...
  [2000/2542] images processed...
  [2200/2542] images processed...
  [2400/2542] images processed...
  [2542/2542] images processed...
[extract_image_features] Image feature matrix: (2542, 1280)
[combine_features] Combined feature matrix: (2542, 1316)

Train: 2033 | Test: 509
[train_classifier] Trained xgboost on 2033 samples.

MODEL EVALUATION
  Accuracy :  0.8448
  ROC-AUC  :  0.9288

Classification Report:
                  precision    recall  f1-score   support

Non-pregnant (0)       0.82      0.97      0.89       316
    Pregnant (1)       0.92      0.65      0.76       193

        accuracy                           0.84       509
       macro avg       0.87      0.81      0.82   